# 03 Dynamic Agents
Up until now, our Agents have been powerful, but they have been fairly _static_. Fixed tools, fixed prompts, fixed models. _Real-world AI applications_ don't work like that in production! 

For example, we may have to adjust _behavior_ of our Agent depending on whether the end-user is an employee or an external-user OR if her default language preference is different (say Spanish) from our default (say English). When it comes time to deploy _real_ Agents, we need agents that can _dynamically adjust_ themselves.

In this notebook we'll cover how you can adjust models, tools, and even models to build your Agent with on-the-fly - even multiple-times during a conversation! To achieve all this, we'll build custom middleware (as you might have guessed already 😎).

In [1]:
from dotenv import load_dotenv
from rich.console import Console

load_dotenv(override=True)
console = Console()

In the first notebook [01_middleware.ipynb](01_middleware.ipynb) of this series, we saw how we used middleware to _trim_ our chat context.

<div align="center"> 
<img src="images/node_style_middleware.png" width="250" heigh="250" alt="Node style middleware"/> 
</div>

Middleware such as the `SummarizationMiddleware`, which gets called before/after agent/model/tool calls is called **Node-style middleware**. Node-style middleware is really good at hooking into the various stages of Agent execution and changes its state - great for use-cases like modifying messages/prompts. 

However, this is _not_ dynamic behavior. To achieve what we are envisaging, we need to tap even deeper and hook into the model or tool instance itself. This isn't _before_ or _after_ the model/tool(s) - this _is_ the model/tool(s) itself. To achieve this, we'll be wrapping our model/tool(s) itself with custom middleware functions, which are aptly named _wrap-style_ middleware. The middleware functions we define for our will be annotated with `@wrap_model_call`/`@awrap_model_call` or `@wrap_tool_call`/`@awrap_tool_call` annotations.

* The model instance is represented as a _model request_. This contains information like the system prompt, available tool calls, the state and the foundation model itself. Wrapping this with our annotated middleware allows us to do things like swap the underlying Gemini model with Claude at runtime, add specialized tools based on user's permissions (e.g. employee gets more tools (such as connections to all internal sources of information) than a non-employee user (who gets access to _only_ a selected sources of info.)) 



In [2]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest


@dataclass
class LanguageContext:
    user_language: str = "English"  # English by default, or as set by user!


@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """generates dynamic system prompt based on user's preferred language"""
    preferred_language: str = request.runtime.context.user_language
    base_prompt: str = "You are a helpful assistant. "

    if preferred_language.lower().strip() != "english":
        base_prompt = base_prompt + f"ONLY respond in {preferred_language}."

    print(f" ---- dynamic system prompt -> {base_prompt} ---- ")
    return base_prompt

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-5-nano",
    context_schema=LanguageContext,
    middleware=[user_language_prompt],
)

In [4]:
from langchain.messages import HumanMessage

user_message = HumanMessage("Hello, how are you?")

In [5]:
# first attempt - don't set language
response = agent.invoke(
    {"messages": [user_message]},
    # use default value that is set
    context=LanguageContext(),
)
print(response["messages"][-1].content)

 ---- dynamic system prompt -> You are a helpful assistant.  ---- 
Hi there! I’m here and ready to help with whatever you need. I don’t have feelings, but I can assist with questions, explanations, writing, planning, and more. What would you like to do today?


In [6]:
# second attempt - let's try Spanish
response = agent.invoke(
    {"messages": [user_message]},
    context=LanguageContext("Spanish"),
)
print(response["messages"][-1].content)

 ---- dynamic system prompt -> You are a helpful assistant. ONLY respond in Spanish. ---- 
¡Hola! Estoy bien, gracias. ¿Y tú?


In [7]:
# third attempt - let's try Hindi?
response = agent.invoke(
    {"messages": [user_message]},
    context=LanguageContext("Hindi"),
)
print(response["messages"][-1].content)

 ---- dynamic system prompt -> You are a helpful assistant. ONLY respond in Hindi. ---- 
नमस्ते! मैं ठीक हूँ, धन्यवाद। आप कैसे हैं? बताइए, मैं कैसे मदद कर सकता हूँ?


## Chatbot

In this section, let's build a chatbot that is accessed by both internal users (employees) and external users.

The purpose of this example is to show you how we can dynamically control the tools that are available to the model based on whether the user is an internal or external user. Internal users get access to web search as well as database search (pretent this is an internal database). External users get access to ONLY web search. The tools are set dynamically depending on the context provided to agent, which determines if user is _internal_ or _external_.

In [39]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()
db = SQLDatabase.from_uri("sqlite:///db/chinook.db")
db_schema = db.get_table_info()


# define tools for our chatbot
@tool
def web_search(query: str) -> Dict[str, Any]:
    """search the web for information"""
    print(f" --- web_search({query}) tool called --- ")
    return tavily_client.search(query)


@tool
def sql_query(query: str) -> str:
    """obtain information from database using SQL queries"""
    print(f" --- sql_query({query}) tool called --- ")
    try:
        return db.run(query)
    except Exception as e:
        return f"Database error: {e}"

In [40]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable
from dataclasses import dataclass


# call that dynamically assigns the tools a model can use
# depending on whether the user is an internal or external user
@wrap_model_call
def dynamic_tool_call(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """dynamically call tools based on runtime context"""
    user_role = request.runtime.context.user_role
    print(f" --- in dynamic_tool_call -> user_role = {user_role} --- ")

    if user_role.lower().strip() != "internal":
        # external user -> will get access to ONLY web-search
        tools = [web_search]
        # modify the tools used by model
        request = request.override(tools=tools)

    return handler(request)


@dataclass
class UserRole:
    user_role: str = "internal"

In [41]:
# create our agent
agent = create_agent(
    model="openai:gpt-5-nano",
    # by default we get both the tools
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    # schema that decided if user is internal or not
    context_schema=UserRole,
    # system_prompt=system_prompt,
)

In [42]:
from langchain_core.messages import HumanMessage

# ONLY internal users get access to database info
user_query = HumanMessage("How many artists in the database?")

In [43]:
# let's try with internal user
response = agent.invoke(
    {"messages": [user_query]},
    context={"user_role": "internal"},
)
print(response["messages"][-1].content)

 --- in dynamic_tool_call -> user_role = internal --- 
 --- sql_query(SELECT COUNT(*) AS artist_count FROM artists;) tool called --- 
 --- in dynamic_tool_call -> user_role = internal --- 
There are 275 artists in the database.


In [44]:
# let's try same query, but as an external user
response = agent.invoke(
    {"messages": [user_query]},
    context={"user_role": "external"},
)
print(response["messages"][-1].content)

 --- in dynamic_tool_call -> user_role = external --- 
I don’t have access to your database, so I can’t tell you the exact number. If you share the database system and the relevant table, I can give you the exact query. In the meantime, here are common ways to count artists:

- SQL (assuming a table named artists)
  - Total rows: SELECT COUNT(*) AS total_artists FROM artists;
  - Distinct artists by id (in case of duplicates): SELECT COUNT(DISTINCT artist_id) AS total_artists FROM artists;
  - Active artists only (if you have a status column): SELECT COUNT(*) AS total_artists FROM artists WHERE status = 'active';

- If you want distinct by name: SELECT COUNT(DISTINCT name) AS total_artists FROM artists;

- NoSQL (MongoDB)
  - Total count: db.artists.countDocuments({});
  - Distinct IDs (distinct _id): db.artists.distinct("_id").length

- Python (pandas)
  - Unique artists by id: df['artist_id'].nunique()

If you can share your DB type (PostgreSQL/MySQL/SQLite, MongoDB, etc.) and the ta

### Switching Models
In this example we show how we can dynamically change the underlying model (language model) used by our Agent./